In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.payer360_master

In [0]:
select * from com_edp_prd.com_intgr.payer_policy_details

In [0]:
WITH policy_names AS (
    SELECT DISTINCT
        CASE
            WHEN UPPER(payer) RLIKE 'UNITED|OPTUM'
                THEN 'UHC/Optum'

            WHEN UPPER(payer) RLIKE 'AETNA|CVS'
                THEN 'Aetna/CVS'

            WHEN UPPER(payer) RLIKE 'CIGNA|ESI|EVERNORTH'
                THEN 'Cigna/ESI'

            WHEN UPPER(payer) RLIKE 'ANTHEM|ELEVANCE|CARELON'
                THEN 'Elevance/Carelon'

            WHEN UPPER(payer) RLIKE 'ILLINOIS|TEXAS|OKLAHOMA|NEW MEXICO'
                THEN 'Prime Therapeutics / HCSC'

            ELSE payer
        END AS payer_name
    FROM com_edp_prd.com_intgr.payer_policy_details
),

master_names AS (
    SELECT DISTINCT payer_name
    FROM com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level
)

SELECT
    COALESCE(p.payer_name, m.payer_name) AS payer_name,
    CASE
        WHEN p.payer_name IS NOT NULL AND m.payer_name IS NOT NULL
            THEN 'Match'
        WHEN p.payer_name IS NOT NULL
            THEN 'Only in payer_policy_details'
        WHEN m.payer_name IS NOT NULL
            THEN 'Only in payer_master_patient_level'
    END AS match_status
FROM policy_names p
FULL OUTER JOIN master_names m
    ON UPPER(TRIM(p.payer_name)) = UPPER(TRIM(m.payer_name))
ORDER BY match_status, payer_name;

In [0]:
select distinct payer
from com_edp_prd.com_intgr.payer_policy_details

In [0]:
SELECT DISTINCT
    CASE
        -- Big 3
        WHEN UPPER(payer) RLIKE 'UNITED|OPTUM|UHC'
            THEN 'UHC / Optum'

        WHEN UPPER(payer) RLIKE 'AETNA|CVS'
            THEN 'Aetna / CVS'

        WHEN UPPER(payer) RLIKE 'CIGNA|ESI|EXPRESS SCRIPTS|EVERNORTH|OXFORD'
            THEN 'Cigna / Evernorth'

        -- Elevance
        WHEN UPPER(payer) RLIKE 'ANTHEM|ELEVANCE|CARELON|WELLPOINT'
            THEN 'Elevance (Anthem)'

        -- HCSC Blues
        WHEN UPPER(payer) RLIKE 'BCBS (ILLINOIS|TEXAS|OKLAHOMA)'
            THEN 'HCSC (BCBS)'

        -- Other Blues
        WHEN UPPER(payer) RLIKE 'BCBS|BLUE CROSS|BLUE SHIELD|HIGHMARK|HORIZON|CARE FIRST|INDEPENDENCE|FLORIDA BLUE|WELLMARK|PREMERA|REGENCE'
            THEN 'BCBS (Non-HCSC)'

        -- Kaiser
        WHEN UPPER(payer) RLIKE 'KAISER'
            THEN 'Kaiser Permanente'

        -- Medicare MAC
        WHEN UPPER(payer) RLIKE 'NOVITAS|NORIDIAN|PALMETTO|CGS|WPS|FIRST COAST|NGS|NATIONAL GOVERNMENT SERVICES'
            THEN 'Medicare MAC'

        -- Medicaid FFS
        WHEN UPPER(payer) RLIKE 'MEDICAID.*FFS'
            THEN 'Medicaid FFS'

        -- Centene
        WHEN UPPER(payer) RLIKE 'WELLCARE|CENTENE|FIDELIS|SUNSHINE|PEACH STATE|SUPERIOR|HEALTHNET'
            THEN 'Centene'

        -- Molina
        WHEN UPPER(payer) RLIKE 'MOLINA'
            THEN 'Molina'

        -- Regional
        WHEN UPPER(payer) RLIKE 'UPMC|HEALTHFIRST|PRIORITY HEALTH|HEALTHPARTNERS|SENTARA|HARVARD PILGRIM|MEDICA|SELECTHEALTH'
            THEN 'Regional Plans'

        ELSE payer
    END AS payer_group
FROM com_edp_prd.com_intgr.payer_policy_details;

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level

In [0]:
select a.payer_name, a.payer_id, b.payer, b.unique_payer_id
from com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level a
left join com_edp_prd.com_intgr.payer_policy_details b 
on a.PAYER_ID = b.unique_payer_id

In [0]:
select a.payer_name, a.payer_id, b.payer, b.unique_payer_id
from com_edp_prd.com_raw.kom_plans a
left join com_edp_prd.com_intgr.payer_policy_details b 
on a.PAYER_ID = b.unique_payer_id